# Task 4.1

In [ ]:
!pip install -q google-genai pydantic

In [ ]:
from google import genai
from google.genai import types
import json

API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

Paste your Gemini API key: AQ.Ab8RN6IhxD6czY8hOtoMyMjxu6loHN3vjL1MAhanUXdettqs4Q
Gemini client initialized successfully!


In [ ]:
MODEL_NAME = "gemini-3.5-flash"
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

In [ ]:
SYSTEM_TEXT = """
You are a healthcare call-centre analysis assistant.
Analyse patient follow-up notes accurately.
Do not invent information or assume reasons that are not stated.
Return a response that follows the provided JSON schema.
"""


def build_prompt(note_text):
    """
    Builds a CRAFT-based prompt for analysing a patient follow-up note.

    Parameters:
        note_text (str): The patient's call-centre follow-up note.

    Returns:
        str: A formatted prompt containing context, role, action,
             tone, format instructions and the patient note.
    """

    prompt = f"""
    CONTEXT:
    A clinic wants to understand why patients miss appointments.

    ROLE:
    You are a healthcare call-centre analyst.

    ACTION:
    Read the patient follow-up note and identify the most likely
    reason for the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    If the reason is unclear or does not fit the categories,
    select other. Do not guess.

    TONE:
    Be factual, neutral and concise.

    FORMAT:
    Return exactly these four fields:
    root_cause, confidence, summary, recommended_action.

    The confidence must be between 0 and 1.
    The summary MUST contain no more than 25 words.
    Count the words before responding.
    Do not exceed this limit under any circumstances.

    PATIENT NOTE:
    {note_text}
    """

    return prompt

In [ ]:
total_api_calls = 0

In [ ]:
LAST_RAW_RESPONSE = ""

def analyse_note(note_text):
    """
    Sends one patient follow-up note to Gemini for structured analysis.

    Parameters:
        note_text (str): The patient's call-centre follow-up note.

    Returns:
        dict: Parsed JSON containing exactly four fields:
            - root_cause
            - confidence
            - summary
            - recommended_action

    Process:
        1. Builds the CRAFT prompt.
        2. Sends it to the Gemini Flash model.
        3. Receives and preserves the original JSON response.
        4. Converts the JSON response into a Python dictionary.
    """

    global LAST_RAW_RESPONSE
    LAST_RAW_RESPONSE = ""

    global total_api_calls
    total_api_calls += 1

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    LAST_RAW_RESPONSE = response.text

    data = json.loads(response.text)

    return data


In [ ]:
note_n001 = "Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week."

result_n001 = analyse_note(note_n001)

print(result_n001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she forgot about it after receiving no reminder message. She requested to reschedule.', 'recommended_action': "Reschedule the patient's appointment for next week and verify her contact details to ensure future SMS or email reminders are properly received."}


In [ ]:
summary_words = len(result_n001["summary"].split())

print("Summary word count:", summary_words)
print("Within 25-word limit:", summary_words <= 25)

Summary word count: 18
Within 25-word limit: True


# Task 4.2

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)

In [ ]:
data = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.95,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send appointment reminder"
}

In [ ]:
insight = FollowupInsight(**data)

print(insight)

root_cause='forgot_or_no_reminder' confidence=0.95 summary='Patient forgot appointment' recommended_action='Send appointment reminder'


In [ ]:
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

In [ ]:
def validate_root_cause(root_cause):
    """
    Checks whether a root cause belongs to the approved categories.

    Parameters:
        root_cause (str): The root cause returned by Gemini.

    Returns:
        bool: True if the category is allowed, otherwise False.
    """

    return root_cause in ALLOWED_ROOT_CAUSES

print(validate_root_cause("forgot_or_no_reminder"))
print(validate_root_cause("personal_reason"))

True
False


In [ ]:
import pandas as pd

df_notes = pd.read_csv("/content/followup_notes.csv")

print(df_notes.head(10))
print(df_notes.columns)
print(df_notes.shape)

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...
5    N006          A0507  Manager called an urgent meeting at the office...
6    N007          A0333  Bhool gaya, SMS nahi aaya tha. Dobara booking ...
7    N008          A0222  City bus strike that day, no auto available ne...
8    N009          A0977  Phone was switched off, could not reach the pa...
9    N010          A0596  Had a shift change at the factory and could no...
Index(['note_id', 'appointment_id', 'note_text'], dtype='object')
(30, 3)


In [ ]:
df_first10 = df_notes.head(10).copy()

print(df_first10.shape)

(10, 3)


In [ ]:
for _, row in df_first10.iterrows():

    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":
        print(row["note_id"], "Blank note")
    else:
        print(row["note_id"], "Note available")

N001 Note available
N002 Note available
N003 Note available
N004 Blank note
N005 Note available
N006 Note available
N007 Note available
N008 Note available
N009 Note available
N010 Note available


In [ ]:
results = []

In [ ]:
for _, row in df_first10.iterrows():

    note_text = row["note_text"]

    if pd.isna(note_text) or str(note_text).strip() == "":

        results.append({
            "note_id": row["note_id"],
            "appointment_id": row["appointment_id"],
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        })

In [ ]:
print(results)
print("Total records stored:", len(results))

[{'note_id': 'N004', 'appointment_id': 'A0926', 'root_cause': 'other', 'confidence': 0.0, 'summary': 'No note recorded', 'recommended_action': 'Re-call patient', 'validation_status': 'skipped_blank', 'raw_response': ''}]
Total records stored: 1


In [ ]:
import time

results = []
api_calls = 0


for _, row in df_first10.iterrows():

    note_text = row["note_text"]


    data = {}
    raw_response = ""

    if pd.isna(note_text) or str(note_text).strip() == "":


        root_cause = "other"
        confidence = 0.0
        summary = "No note recorded"
        recommended_action = "Re-call patient"
        validation_status = "skipped_blank"

    else:


        if api_calls > 0:
            time.sleep(13)


        api_calls = api_calls + 1


        try:


            data = analyse_note(note_text)


            raw_response = LAST_RAW_RESPONSE


            insight = FollowupInsight(**data)


            if validate_root_cause(insight.root_cause):
                validation_status = "valid"
            else:
                validation_status = "invalid"


            root_cause = data.get("root_cause", "")
            confidence = data.get("confidence", "")
            summary = data.get("summary", "")
            recommended_action = data.get("recommended_action", "")


        except Exception as e:

            raw_response = LAST_RAW_RESPONSE

            validation_status = "invalid"


            root_cause = data.get("root_cause", "")
            confidence = data.get("confidence", "")
            summary = data.get("summary", "")
            recommended_action = data.get("recommended_action", "")

            print("Error in", row["note_id"], ":", e)


    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": root_cause,
        "confidence": confidence,
        "summary": summary,
        "recommended_action": recommended_action,
        "validation_status": validation_status,
        "raw_response": raw_response
    }

    results.append(result)



print("Total records:", len(results))
print("Total API calls:", api_calls)


Total records: 10
Total API calls: 9


In [ ]:
results

[{'note_id': 'N001',
  'appointment_id': 'A0872',
  'root_cause': 'forgot_or_no_reminder',
  'confidence': 0.95,
  'summary': 'Patient completely forgot about the visit because she never received a reminder message. She requested to rebook for next week.',
  'recommended_action': 'Reschedule the patient for next week and verify that her phone number and reminder preferences are correct in the system.',
  'validation_status': 'valid',
  'raw_response': '{"root_cause":"forgot_or_no_reminder","confidence":0.95,"summary":"Patient completely forgot about the visit because she never received a reminder message. She requested to rebook for next week.","recommended_action":"Reschedule the patient for next week and verify that her phone number and reminder preferences are correct in the system."}'},
 {'note_id': 'N002',
  'appointment_id': 'A0897',
  'root_cause': 'distance_or_transport',
  'confidence': 0.95,
  'summary': 'The patient missed the appointment due to living far from the clinic an

In [ ]:
df_results = pd.DataFrame(results)

# print(df_results.head(10))
print(df_results.shape)

df_results.to_csv("notes_analysed.csv", index=False)

(10, 8)


In [ ]:
print("Shape:", df_results.shape)
print("Columns:", df_results.columns.tolist())

print(df_results["validation_status"].value_counts())

Shape: (10, 8)
Columns: ['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary', 'recommended_action', 'validation_status', 'raw_response']
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


# Task 4.3

In [ ]:
bad_data_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot appointment",
    "recommended_action": "Send appointment reminder"
}

try:
    FollowupInsight(**bad_data_1)

except ValidationError as e:
    print(e)

1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [ ]:
bad_data_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.85,
    "summary": "Consultation fee too high"
}

try:
    FollowupInsight(**bad_data_2)

except ValidationError as e:
    print(e)

1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...sultation fee too high'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [ ]:
def compare_prompt(prompt):
    """
    Sends a prompt to Gemini using the same settings as Task 4.1.

    Parameters:
        prompt (str): The prompt being tested (V1 or V2).

    Returns:
        dict: Parsed JSON response from Gemini.

    """

    global total_api_calls
    total_api_calls += 1

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return json.loads(response.text)

In [ ]:
note_7 = df_notes.loc[
    df_notes["note_id"] == "N007", "note_text"
].iloc[0]

note_9 = df_notes.loc[
    df_notes["note_id"] == "N009", "note_text"
].iloc[0]

print("N007:", note_7)
print("N009:", note_9)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.
N009: Phone was switched off, could not reach the patient after three attempts.


In [ ]:
import time

note_text = note_7

prompt_v1 = "Classify this note: " + note_text
prompt_v2 = build_prompt(note_text)

v1_n007 = compare_prompt(prompt_v1)
time.sleep(13)

v2_n007 = compare_prompt(prompt_v2)
time.sleep(13)

note_text = note_9

prompt_v1 = "Classify this note: " + note_text
prompt_v2 = build_prompt(note_text)


v1_n009 = compare_prompt(prompt_v1)
time.sleep(13)

v2_n009 = compare_prompt(prompt_v2)


print("All four API calls completed.")
print("N007 V1:", v1_n007)
print("N007 V2:", v2_n007)
print("N009 V1:", v1_n009)
print("N009 V2:", v2_n009)

All four API calls completed.
N007 V1: {'root_cause': 'Patient forgot and did not receive an SMS reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot about it and claimed they did not receive an SMS reminder. They requested to rebook.', 'recommended_action': "Reschedule the patient's appointment and verify/update their phone number in the system to ensure SMS reminders are successfully delivered."}
N007 V2: {'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'The patient missed the appointment because they forgot and did not receive an SMS reminder. They want to rebook.', 'recommended_action': "Reschedule the patient's appointment and verify that their phone number is correctly updated in the SMS reminder system."}
N009 V1: {'root_cause': 'Patient unreachable - Phone switched off', 'confidence': 0.98, 'summary': 'The patient could not be reached after three attempts because their phone was switched off.', 'recommended_acti

The V2 system is chosen for deployment because the V2 system consistently follows the approved root‑cause categories. In the case of N007, the V2 system correctly classified the note as forgot_or_no_reminder, while the V1 system returned a category. For N009, the V2 system correctly classified the note as other, whereas the V1 system generated a new category. Therefore, the CRAFT‑based V2 prompt provides reliable and standardized outputs for downstream analysis.

In [ ]:
print("Total Gemini API calls:", total_api_calls)

Total Gemini API calls: 14


# Task 4.4

In [ ]:
import pandas as pd


df_results = pd.read_csv("/content/notes_analysed.csv")
df_appointments = pd.read_csv("/content/appointments_clean.csv")

df_merged = pd.merge(
    df_results,
    df_appointments,
    on="appointment_id",
    how="left",
    validate="many_to_one"
)

display(df_merged)

print("Merged table shape:", df_merged.shape)
print("Total rows:", len(df_merged))
print(
    "Missing appointment matches:",
    df_merged["reminder_sent"].isna().sum()
)


assert len(df_merged) == 10, "Acceptance criteria failed: expected 10 rows."

print("Task 4.4(a): 10-row acceptance criterion passed.")

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response,patient_id,clinic_id,...,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,N001,A0872,forgot_or_no_reminder,0.95,Patient completely forgot about the visit beca...,Reschedule the patient for next week and verif...,valid,"{""root_cause"":""forgot_or_no_reminder"",""confide...",P144,C05,...,F,17.6,750,App,Consultation,No,1,No-Show,14,1
1,N002,A0897,distance_or_transport,0.95,The patient missed the appointment due to livi...,Offer telehealth options or assist with schedu...,valid,"{""root_cause"":""distance_or_transport"",""confide...",P268,C07,...,M,17.2,1219,Phone,Follow-up,No,0,No-Show,5,1
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment because the co...,Provide information on available financial ass...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,...",P006,C04,...,F,5.3,800,App,Consultation,Yes,0,No-Show,34,1
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,NaN,P449,C04,...,M,4.5,300,Phone,Follow-up,No,0,No-Show,40,1
4,N005,A0534,illness_or_emergency,0.95,The patient missed the appointment due to a hi...,Follow up with the patient in a few days to re...,valid,"{""root_cause"":""illness_or_emergency"",""confiden...",P400,C04,...,F,4.2,550,App,Follow-up,Yes,0,No-Show,26,1
5,N006,A0507,work_or_schedule,0.95,Patient missed the appointment due to an urgen...,Reschedule the patient for an evening appointm...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":...",P230,C04,...,F,0.8,700,App,Consultation,Yes,2,No-Show,28,1
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because no SMS ...,Reschedule the appointment and verify the pati...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ...",P472,C08,...,F,15.2,850,Walk-In,Consultation,No,2,No-Show,9,1
7,N008,A0222,distance_or_transport,0.95,Patient missed the appointment due to a city b...,Offer telehealth options or assist with altern...,valid,"{\n ""root_cause"": ""distance_or_transport"",\n ...",P125,C02,...,F,15.8,650,App,Follow-up,Yes,1,No-Show,4,1
8,N009,A0977,other,0.90,The patient could not be reached after three p...,Send an automated text message or email to pro...,valid,"{""root_cause"":""other"",""confidence"":0.9,""summar...",P334,C01,...,M,3.6,800,App,Consultation,Yes,1,No-Show,4,1
9,N010,A0596,work_or_schedule,0.95,Patient missed the appointment due to an unexp...,Reschedule the appointment for an available Sa...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con...",P297,C04,...,F,10.1,400,App,Follow-up,No,1,No-Show,21,1


Merged table shape: (10, 23)
Total rows: 10
Missing appointment matches: 0
Task 4.4(a): 10-row acceptance criterion passed.


In [ ]:
root_cause_reminder_table = pd.crosstab(
    df_merged["root_cause"],
    df_merged["reminder_sent"],
    margins=True
)

print("Root Cause vs Reminder Sent")
display(root_cause_reminder_table)

Root Cause vs Reminder Sent


reminder_sent,No,Yes,All
root_cause,,,
cost_or_fee,0,1,1
distance_or_transport,1,1,2
forgot_or_no_reminder,2,0,2
illness_or_emergency,0,1,1
other,1,1,2
work_or_schedule,1,1,2
All,5,5,10


In [ ]:
df_merged.to_csv("task_4_4_merged.csv", index=False)

root_cause_reminder_table.to_csv("task_4_4_crosstab.csv")

print("Saved task_4_4_merged.csv")
print("Saved task_4_4_crosstab.csv")

Saved task_4_4_merged.csv
Saved task_4_4_crosstab.csv


In [ ]:
print("Root cause counts, including missing values:")
print(df_merged["root_cause"].value_counts(dropna=False))

print("\nValidation status:")
print(df_merged["validation_status"].value_counts(dropna=False))

print("\nRoot cause and reminder status:")
display(
    df_merged[
        ["note_id", "root_cause", "validation_status", "reminder_sent"]
    ]
)

Root cause counts, including missing values:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Validation status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64

Root cause and reminder status:


,note_id,root_cause,validation_status,reminder_sent
0,N001,forgot_or_no_reminder,valid,No
1,N002,distance_or_transport,valid,No
2,N003,cost_or_fee,valid,Yes
3,N004,other,skipped_blank,No
4,N005,illness_or_emergency,valid,Yes
5,N006,work_or_schedule,valid,Yes
6,N007,forgot_or_no_reminder,valid,No
7,N008,distance_or_transport,valid,Yes
8,N009,other,valid,Yes
9,N010,work_or_schedule,valid,No
